In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn import svm, datasets
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score,
f1_score, classification_report,ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

In [ ]:
data = pd.read_csv("/content/sample_data/mushrooms-full-dataset.csv")

In [ ]:
data.head()

In [ ]:
data['poisonous'].value_counts()

In [ ]:
data.isnull().sum()

In [ ]:
target = data['poisonous']
inputs = data.drop('poisonous',axis=1)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(inputs,target,test_size=0.2,random_state = 365, stratify = target)

In [ ]:
y_train.value_counts()

In [ ]:
y_test.value_counts()

In [ ]:
enc_i = OrdinalEncoder()
enc_t = LabelEncoder()

In [ ]:
x_train_transf = enc_i.fit_transform(x_train)
x_test_transf = enc_i.transform(x_test)

y_train_transf = enc_t.fit_transform(y_train)
y_test_transf = enc_t.transform(y_test)

In [ ]:
y_train_transf

In [ ]:
x_train_transf

In [ ]:
scaling = MinMaxScaler(feature_range = (-1,1)).fit(x_train_transf)
x_train_rescaled = scaling.transform(x_train_transf)

In [ ]:
x_train_rescaled

In [ ]:
C = 1.0
svc = svm.SVC(kernel='linear', C=C).fit(x_train_rescaled, y_train_transf)

In [ ]:
y_pred_test = svc.predict(scaling.transform(x_test_transf))

In [ ]:
fig, ax = plt.subplots(figsize = (8,5))

cmp = ConfusionMatrixDisplay( confusion_matrix(y_test_transf, y_pred_test), display_labels = enc_t.classes_)
cmp.plot(ax = ax);

In [ ]:
print(classification_report(y_test_transf, y_pred_test, target_names = ['Edible', 'Poisonous']))

In [ ]:
tuned_parameters = [ {"kernel":["linear"], "C":[1,10,20]},
                      {"kernel":["poly"], "C":[1,10,20]},
                    {"kernel":["rbf"],"gamma":[1e-3,1e-4] , "C":[1,10,20]}
                     ]

In [ ]:
scores = ["precision", "recall"]

In [ ]:
for score in scores:
    print("# Tuning hyper-parameters for %s" % score)
    print()

    clf = GridSearchCV(svm.SVC(), tuned_parameters, scoring="%s_macro" % score)
    clf.fit(x_train_rescaled, y_train_transf)

    print("Best parameters set found on development set:")
    print()
    print(clf.best_params_)
    print()
    print("Grid scores on development set:")
    print()
    means = clf.cv_results_["mean_test_score"]
    stds = clf.cv_results_["std_test_score"]
    for mean, std, params in zip(means, stds, clf.cv_results_["params"]):
        print("%0.3f (+/-%0.03f) for %r" % (mean, std * 2, params))
    print()

    print("Detailed classification report:")
    print()
    print("The model is trained on the full development set.")
    print("The scores are computed on the full evaluation set.")
    print()
    y_true, y_pred = y_test_transf, clf.predict(scaling.transform(x_test_transf))
    print(classification_report(y_true, y_pred))
    print()